<a href="https://colab.research.google.com/github/fatmasenguler/Spanning-Tree_Thermostatics_of_Allostery/blob/main/1_channel_convergence_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install biopython networkx pandas matplotlib numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 33.7 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving 6GOD.pdb to 6GOD.pdb
Saving 6GOF.pdb to 6GOF.pdb


In [ ]:
import os
import sys
import math
import numpy as np
import networkx as nx
from Bio.PDB import PDBParser, NeighborSearch
from numpy.linalg import pinv, slogdet, LinAlgError

# ─────────────────────────────────────────────────────────────
# GRAPH CONSTRUCTION
# ─────────────────────────────────────────────────────────────
def build_ca_graph(pdb_file, cutoff):
    """Return (G, res_ids, edge_weight_dict) for CA contact graph."""
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("prot", pdb_file)
    model = structure[0]

    chosen_chain = None
    for chain in model.get_chains():
        if any("CA" in res for res in chain):
            chosen_chain = chain
            break
    if chosen_chain is None:
        raise ValueError(f"No chain with CA atoms found in {pdb_file}.")

    ca_atoms, res_ids = [], []
    atom_to_resid = {}
    for res in chosen_chain:
        if "CA" in res:
            atom = res["CA"]
            rid = res.get_id()[1]
            ca_atoms.append(atom)
            res_ids.append(rid)
            atom_to_resid[id(atom)] = rid

    G = nx.Graph()
    G.add_nodes_from(res_ids)

    ns = NeighborSearch(ca_atoms)
    for a1, a2 in ns.search_all(cutoff, level="A"):
        r1 = atom_to_resid[id(a1)]
        r2 = atom_to_resid[id(a2)]
        if r1 == r2:
            continue
        d = float(np.linalg.norm(a1.coord - a2.coord))
        G.add_edge(r1, r2, weight=d)

    edge_weight = {
        (min(u, v), max(u, v)): float(data["weight"])
        for u, v, data in G.edges(data=True)
    }
    return G, res_ids, edge_weight


# ─────────────────────────────────────────────────────────────
# LAPLACIAN / PSEUDOINVERSE HELPERS
# ─────────────────────────────────────────────────────────────
def build_laplacian(G, res_ids, index_map, kT):
    """Weighted Laplacian with w_ij = exp(-d_ij / kT)."""
    n = len(res_ids)
    L = np.zeros((n, n), dtype=float)
    for u, v, data in G.edges(data=True):
        w = math.exp(-data["weight"] / kT)
        i, j = index_map[u], index_map[v]
        L[i, i] += w
        L[j, j] += w
        L[i, j] -= w
        L[j, i] -= w
    return L


def effective_resistance(L_pinv, index_map, src, dst):
    """R_ab = K_aa + K_bb - 2*K_ab."""
    a, b = index_map[src], index_map[dst]
    return float(L_pinv[a, a] + L_pinv[b, b] - 2.0 * L_pinv[a, b])


def global_K(G, res_ids, index_map, kT):
    """Return Moore-Penrose pseudoinverse of weighted Laplacian."""
    L = build_laplacian(G, res_ids, index_map, kT)
    return pinv(L)


# ─────────────────────────────────────────────────────────────
# PATH ENUMERATION FOR A GIVEN MAX_LENGTH
# ─────────────────────────────────────────────────────────────
def enumerate_paths_up_to_length(G, src, dst, max_L, adjacency_cache):
    """
    Enumerate ALL simple paths from src to dst with node length <= max_L.
    Returns: list of paths
    """
    all_paths = []

    for L in range(2, max_L + 1):
        target_depth = L - 1
        stack = [(src, [src], {src})]

        while stack:
            node, path, visited = stack.pop()
            depth = len(path) - 1

            if depth == target_depth:
                if node == dst:
                    all_paths.append(list(path))
                continue

            for nb in adjacency_cache.get(node, []):
                if nb not in visited:
                    stack.append((nb, path + [nb], visited | {nb}))

    return all_paths


def get_subgraph_from_paths(G, all_paths):
    """Build subgraph containing ALL nodes from all paths."""
    all_nodes = set()
    for path in all_paths:
        all_nodes.update(path)

    if len(all_nodes) < 2:
        return None, [], {}

    sub = G.subgraph(all_nodes).copy()
    sub_nodes = list(sub.nodes())
    sub_idx = {n: i for i, n in enumerate(sub_nodes)}

    return sub, sub_nodes, sub_idx


def compute_resistance_for_max_L(G, res_ids, index_map, src, dst, max_L, kT):
    """
    Compute effective resistance using ALL paths with length <= max_L.
    Returns: (R_eff, n_paths, n_nodes_in_subgraph)
    """
    adjacency = {n: list(G.neighbors(n)) for n in G.nodes()}

    # Enumerate all paths up to max_L
    all_paths = enumerate_paths_up_to_length(G, src, dst, max_L, adjacency)
    n_paths = len(all_paths)

    if n_paths == 0:
        return None, 0, 0

    # Build subgraph containing all these paths
    sub, sub_nodes, sub_idx = get_subgraph_from_paths(G, all_paths)

    if sub is None or not nx.has_path(sub, src, dst):
        return None, n_paths, len(sub_nodes) if sub else 0

    # Compute resistance on this subgraph
    L_sub = build_laplacian(sub, sub_nodes, sub_idx, kT)
    K_sub = pinv(L_sub)
    R_sub = effective_resistance(K_sub, sub_idx, src, dst)

    if not np.isfinite(R_sub) or R_sub <= 0:
        return None, n_paths, len(sub_nodes)

    return R_sub, n_paths, len(sub_nodes)


# ─────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────
def main():
    print("=" * 60)
    print("  KRAS Convergence Analysis")
    print("  R(L_max) as function of maximum path length")
    print("=" * 60)

    # Get input files
    wt_pdb = input("\nWT PDB file  (e.g. 6GOD.pdb): ").strip()
    mut_pdb = input("G12D PDB file (e.g. 6GOF.pdb): ").strip()

    for f in (wt_pdb, mut_pdb):
        if not os.path.isfile(f):
            sys.exit(f"File not found: {f}")

    # Get parameters
    cutoff = float(input("CA cutoff (A)   [8.0]: ").strip() or "8.0")
    kT = float(input("kT              [1.0]: ").strip() or "1.0")

    # Get channel residues
    src = int(input("Source residue number: ").strip())
    dst = int(input("Destination residue number: ").strip())
    channel_name = input("Channel name/label (e.g., P-loop): ").strip() or f"Res_{src}_to_{dst}"

    # Get L_max values
    L_input = input("L_max values (comma-separated, e.g., 4,5,6,7,8,9,10) [4,5,6,7,8,9,10]: ").strip()
    if L_input:
        max_L_values = [int(x.strip()) for x in L_input.split(',')]
    else:
        max_L_values = [4, 5, 6, 7, 8, 9, 10]

    max_L_values = sorted(set(max_L_values))
    print(f"\n  Will compute for L_max = {max_L_values}")

    if cutoff <= 0 or kT <= 0:
        sys.exit("Cutoff and kT must be positive.")

    # Build graphs
    print("\n-- Building CA contact graphs --")
    G_wt, res_wt, ew_wt = build_ca_graph(wt_pdb, cutoff)
    G_mut, res_mut, ew_mut = build_ca_graph(mut_pdb, cutoff)

    idx_wt = {r: i for i, r in enumerate(res_wt)}
    idx_mut = {r: i for i, r in enumerate(res_mut)}

    print(f"  WT  : {len(res_wt)} residues, {G_wt.number_of_edges()} edges")
    print(f"  G12D: {len(res_mut)} residues, {G_mut.number_of_edges()} edges")

    # Check if residues exist
    print("\n-- Validating residues --")
    if src not in idx_wt or dst not in idx_wt:
        print(f"  ERROR: Residues {src} or {dst} not found in WT structure!")
        print(f"  Available residues: {min(res_wt)} to {max(res_wt)}")
        sys.exit("Invalid residues for WT")

    if src not in idx_mut or dst not in idx_mut:
        print(f"  ERROR: Residues {src} or {dst} not found in G12D structure!")
        print(f"  Available residues: {min(res_mut)} to {max(res_mut)}")
        sys.exit("Invalid residues for G12D")

    print(f"  OK: residues {src} and {dst} found in both structures")

    # Compute full graph resistances (ground truth)
    print("\n-- Computing full graph resistances (ground truth) --")
    K_wt_full = global_K(G_wt, res_wt, idx_wt, kT)
    K_mut_full = global_K(G_mut, res_mut, idx_mut, kT)

    R_full_wt = effective_resistance(K_wt_full, idx_wt, src, dst)
    R_full_mut = effective_resistance(K_mut_full, idx_mut, src, dst)

    print(f"  WT   full graph R({src}->{dst}) = {R_full_wt:.8f}")
    print(f"  G12D full graph R({src}->{dst}) = {R_full_mut:.8f}")

    # Convergence analysis
    print("\n-- Convergence analysis --")
    print("  Computing R(L_max) for each L_max...")

    # Open output file
    output_file = f"convergence_{channel_name.replace(' ', '_')}.txt"

    with open(output_file, 'w') as f:
        f.write("=" * 80 + "\n")
        f.write("KRAS Convergence Analysis\n")
        f.write(f"Channel: {channel_name} (residues {src} -> {dst})\n")
        f.write(f"WT PDB: {wt_pdb}\n")
        f.write(f"G12D PDB: {mut_pdb}\n")
        f.write(f"Cutoff: {cutoff} A\n")
        f.write(f"kT: {kT}\n")
        f.write(f"Date: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("=" * 80 + "\n\n")

        f.write("FULL GRAPH RESULTS (ground truth):\n")
        f.write(f"  WT   R({src}->{dst}) = {R_full_wt:.8f}\n")
        f.write(f"  G12D R({src}->{dst}) = {R_full_mut:.8f}\n\n")

        f.write("CONVERGENCE RESULTS:\n")
        f.write("R(L_max) = effective resistance using all paths with length <= L_max\n")
        f.write("ratio = R(L_max) / R_full\n\n")

        f.write("-" * 100 + "\n")
        f.write(f"{'L_max':<8} {'Structure':<10} {'R(L_max)':>15} {'Ratio':>12} {'# Paths':>12} {'# Nodes':>12}\n")
        f.write("-" * 100 + "\n")

        # Print header to screen
        print("\n" + "-" * 80)
        print(f"{'L_max':<8} {'Structure':<10} {'R(L_max)':>15} {'Ratio':>12} {'# Paths':>12} {'# Nodes':>12}")
        print("-" * 80)

        # WT analysis
        for L_max in max_L_values:
            print(f"  WT L_max={L_max:2d}...", end=" ", flush=True)
            R_sub, n_paths, n_nodes = compute_resistance_for_max_L(
                G_wt, res_wt, idx_wt, src, dst, L_max, kT
            )

            if R_sub is not None:
                ratio = R_sub / R_full_wt
                f.write(f"{L_max:<8} {'WT':<10} {R_sub:>15.8f} {ratio:>12.8f} {n_paths:>12} {n_nodes:>12}\n")
                print(f"R={R_sub:.6f}, ratio={ratio:.6f}, paths={n_paths}, nodes={n_nodes}")
            else:
                f.write(f"{L_max:<8} {'WT':<10} {'N/A':>15} {'N/A':>12} {n_paths:>12} {n_nodes:>12}\n")
                print(f"No paths found")

        # G12D analysis
        for L_max in max_L_values:
            print(f"  G12D L_max={L_max:2d}...", end=" ", flush=True)
            R_sub, n_paths, n_nodes = compute_resistance_for_max_L(
                G_mut, res_mut, idx_mut, src, dst, L_max, kT
            )

            if R_sub is not None:
                ratio = R_sub / R_full_mut
                f.write(f"{L_max:<8} {'G12D':<10} {R_sub:>15.8f} {ratio:>12.8f} {n_paths:>12} {n_nodes:>12}\n")
                print(f"R={R_sub:.6f}, ratio={ratio:.6f}, paths={n_paths}, nodes={n_nodes}")
            else:
                f.write(f"{L_max:<8} {'G12D':<10} {'N/A':>15} {'N/A':>12} {n_paths:>12} {n_nodes:>12}\n")
                print(f"No paths found")

        f.write("-" * 100 + "\n\n")

        # Summary table for easy copy-paste to Origin
        f.write("SUMMARY TABLE FOR ORIGIN (tab-separated):\n")
        f.write("L_max\tWT_R\tWT_ratio\tWT_paths\tWT_nodes\tG12D_R\tG12D_ratio\tG12D_paths\tG12D_nodes\n")

        for L_max in max_L_values:
            # Get WT data
            R_wt, n_paths_wt, n_nodes_wt = None, 0, 0
            R_sub, n_paths, n_nodes = compute_resistance_for_max_L(
                G_wt, res_wt, idx_wt, src, dst, L_max, kT
            )
            if R_sub is not None:
                R_wt = R_sub
                n_paths_wt = n_paths
                n_nodes_wt = n_nodes
                ratio_wt = R_wt / R_full_wt
            else:
                ratio_wt = None

            # Get G12D data
            R_mut, n_paths_mut, n_nodes_mut = None, 0, 0
            R_sub, n_paths, n_nodes = compute_resistance_for_max_L(
                G_mut, res_mut, idx_mut, src, dst, L_max, kT
            )
            if R_sub is not None:
                R_mut = R_sub
                n_paths_mut = n_paths
                n_nodes_mut = n_nodes
                ratio_mut = R_mut / R_full_mut
            else:
                ratio_mut = None

            wt_str = f"{R_wt:.8f}" if R_wt else "NaN"
            wt_ratio_str = f"{ratio_wt:.8f}" if ratio_wt else "NaN"
            mut_str = f"{R_mut:.8f}" if R_mut else "NaN"
            mut_ratio_str = f"{ratio_mut:.8f}" if ratio_mut else "NaN"

            f.write(f"{L_max}\t{wt_str}\t{wt_ratio_str}\t{n_paths_wt}\t{n_nodes_wt}\t"
                    f"{mut_str}\t{mut_ratio_str}\t{n_paths_mut}\t{n_nodes_mut}\n")

        f.write("=" * 80 + "\n")

    print(f"\n-- Results saved to: {output_file}")
    print("\n====== Analysis complete ======")


if __name__ == "__main__":
    import time
    main()

  KRAS Convergence Analysis
  R(L_max) as function of maximum path length

WT PDB file  (e.g. 6GOD.pdb): 6GOD.pdb
G12D PDB file (e.g. 6GOF.pdb): 6GOF.pdb
CA cutoff (A)   [8.0]: 7.8
kT              [1.0]: 1.0
Source residue number: 6
Destination residue number: 11
Channel name/label (e.g., P-loop): P-loop
L_max values (comma-separated, e.g., 4,5,6,7,8,9,10) [4,5,6,7,8,9,10]: 4,5,6,7,8,9,10

  Will compute for L_max = [4, 5, 6, 7, 8, 9, 10]

-- Building CA contact graphs --
  WT  : 172 residues, 796 edges
  G12D: 172 residues, 803 edges

-- Validating residues --
  OK: residues 6 and 11 found in both structures

-- Computing full graph resistances (ground truth) --
  WT   full graph R(6->11) = 51.97708480
  G12D full graph R(6->11) = 51.61101954

-- Convergence analysis --
  Computing R(L_max) for each L_max...

--------------------------------------------------------------------------------
L_max    Structure         R(L_max)        Ratio      # Paths      # Nodes
----------------------